# B0 — Avellaneda-Stoikov Analytical Baseline

**Purpose:** Cross-validate γ on training days (Jan 1–6); evaluate the fixed-γ A-S agent on the held-out test set (Jan 7–29).

**Design:**
- σ, A, κ are re-estimated per day from that day's data (market parameters, not preference parameters).
- γ is fixed from training-day cross-validation (Optuna TPE, n_trials ≥ 20) — never touched on test data.
- Evaluation: one trajectory per test day (seed=42), consistent with B1–B3.

**Output:** `results/b0_test_results.csv`, `results/b0_gamma_fixed.txt`

In [1]:
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.data_loader import load_multi_day
from procs.gym.calibration import calibrate_from_arrays, tune_gamma
from procs.gym.helpers.fast_rollout import fast_simulate

cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()
print(f"Repo root : {cfg.repo_root}")
print(f"Datasets  : {cfg.datasets_dir}")
print(f"Models    : {cfg.models_dir}")
print(f"Results   : {cfg.results_dir}")

Repo root : C:\Users\john-\Documents\Thesis_AI4T
Datasets  : C:\Users\john-\Documents\Thesis_AI4T\datasets
Models    : C:\Users\john-\Documents\Thesis_AI4T\models
Results   : C:\Users\john-\Documents\Thesis_AI4T\results


## Section 1 — Data Loading and Train/Test Split

In [ ]:
TRAIN_DAYS = 6

daily_S, daily_dt, dates = load_multi_day(str(cfg.datasets_dir), pair=cfg.pair)

train_S     = daily_S[:TRAIN_DAYS]
train_dt    = daily_dt[:TRAIN_DAYS]
train_dates = dates[:TRAIN_DAYS]

test_S      = daily_S[TRAIN_DAYS:]
test_dt     = daily_dt[TRAIN_DAYS:]
test_dates  = dates[TRAIN_DAYS:]

print(f"Training days : {len(train_S)}  ({train_dates[0]} → {train_dates[-1]})")
print(f"Test days     : {len(test_S)}  ({test_dates[0]} → {test_dates[-1]})")
for d, S in zip(train_dates, train_S):
    print(f"  train {d}: {len(S):,} snapshots")

  2025-01-01:  713,815 snapshots, σ=0.000021
  2025-01-02:  766,464 snapshots, σ=0.000035
  2025-01-03:  776,383 snapshots, σ=0.000047
  2025-01-04:  778,293 snapshots, σ=0.000045
  2025-01-05:  723,494 snapshots, σ=0.000031
  2025-01-06:  766,311 snapshots, σ=0.000035
  2025-01-07:  787,093 snapshots, σ=0.000062
  2025-01-08:  821,589 snapshots, σ=0.000052
  2025-01-09:  809,421 snapshots, σ=0.000046
  2025-01-10:  789,320 snapshots, σ=0.000045
  2025-01-11:  724,826 snapshots, σ=0.000023
  2025-01-12:  719,550 snapshots, σ=0.000022
  2025-01-13:  819,981 snapshots, σ=0.000046
  2025-01-14:  775,454 snapshots, σ=0.000038
  SKIP binance_book_snapshot_25_2025-01-15_DOGEUSDT.csv: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.
  2025-01-16:  788,946 snapshots, σ=0.000048
  2025-01-17:  801,259 snapshots, σ=0.000061


## Section 2 — γ Cross-Validation on Training Days

For each training day: calibrate σ, A, κ from that day's data, then run Optuna TPE to find the
γ that maximises Sharpe ratio across 50 fast-simulate trajectories.
The final fixed γ is the mean of per-day optimal values.

In [ ]:
gamma_candidates = []
train_params = []  # (sigma, A, kappa) per training day

for i, (S, dt, date) in enumerate(zip(train_S, train_dt, train_dates)):
    print(f"[{i+1}/{TRAIN_DAYS}] Calibrating {date} ...")
    sigma, A, kappa = calibrate_from_arrays(S, dt, tick_size=cfg.tick_size)
    train_params.append((sigma, A, kappa))
    print(f"  σ={sigma:.6f}  A={A:.4f}  κ={kappa:.0f}")

    best_gamma, _ = tune_gamma(
        S, dt,
        sigma=sigma, kappa=kappa, A=A,
        tick_size=cfg.tick_size,
        Q_MAX=cfg.q_max,
        gamma_range=cfg.as_gamma_range,
        n_trials=cfg.as_gamma_trials,
        num_trajectories=cfg.as_gamma_num_trajectories,
        seed=42,
        verbose=False,
    )
    gamma_candidates.append(best_gamma)
    print(f"  γ_opt = {best_gamma:.4f}")

gamma_fixed = float(np.mean(gamma_candidates))
print(f"\nFixed γ for B0 (mean over {TRAIN_DAYS} training days): {gamma_fixed:.4f}")
print(f"  per-day γ candidates: {[f'{g:.4f}' for g in gamma_candidates]}")

In [ ]:
# Save gamma_fixed
gamma_path = cfg.result_path("b0_gamma_fixed.txt")
with open(gamma_path, "w") as f:
    f.write(str(gamma_fixed))
print(f"Saved γ = {gamma_fixed:.6f} → {gamma_path}")

## Section 3 — Test Evaluation (Days 7–29)

For each test day: re-estimate σ, A, κ from that day's data; apply fixed γ from training.
This mirrors realistic deployment (re-calibrate market parameters each morning, hold γ fixed).

In [ ]:
rows_b0 = []

for i, (S, dt, date) in enumerate(zip(test_S, test_dt, test_dates)):
    sigma, A, kappa = calibrate_from_arrays(S, dt, tick_size=cfg.tick_size)
    T = float(dt.sum())

    stats = fast_simulate(
        midprices=S,
        dt_array=dt,
        gamma=gamma_fixed,
        sigma=sigma,
        kappa=kappa,
        A=A,
        terminal_time=T,
        tick_size=cfg.tick_size,
        Q_MAX=cfg.q_max,
        num_trajectories=1,
        seed=42,
        use_linear_approximation=False,
    )

    rows_b0.append({
        "Day":        str(date),
        "Sharpe":     float(stats["sharpe"].mean()),
        "Sortino":    float(stats["sortino"].mean()),
        "Max DD":     float(stats["max_drawdown"].mean()),
        "P&L-to-MAP": float(stats["pnl_to_map"].mean()),
        "Final PnL":  float(stats["total_pnl"].mean()),
        "Mean |q|": float(stats["mean_abs_q"].mean()),
    })

    if (i + 1) % 5 == 0 or i == len(test_S) - 1:
        print(f"  [{i+1}/{len(test_S)}] {date}  Sharpe={rows_b0[-1]['Sharpe']:.3f}  "
              f"MaxDD={rows_b0[-1]['Max DD']:.4f}")

df_b0 = pd.DataFrame(rows_b0).set_index("Day")
print(f"\nEvaluated {len(df_b0)} test days.")

## Section 4 — Results Display and Save

In [ ]:
pd.set_option("display.float_format", "{:.4f}".format)

summary = pd.DataFrame({
    "Mean":   df_b0.mean(),
    "Std":    df_b0.std(),
    "Median": df_b0.median(),
}).T

print("=== B0 Per-day Test Results ===")
print(df_b0.to_string())
print("\n=== B0 Summary (mean ± std across 23 test days) ===")
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
metrics = ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP"]
for ax, metric in zip(axes.flat, metrics):
    ax.bar(range(len(df_b0)), df_b0[metric].values)
    ax.axhline(df_b0[metric].mean(), color="crimson", linestyle="--", label=f"Mean={df_b0[metric].mean():.3f}")
    ax.set_title(f"B0 — {metric}")
    ax.set_xlabel("Test day index")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.suptitle("B0 Avellaneda-Stoikov: Test Set Metrics (Jan 7–29)", fontsize=13)
plt.tight_layout()
plt.savefig(str(cfg.result_path("b0_test_metrics.png")), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
out_path = cfg.result_path("b0_test_results.csv")
df_b0.to_csv(out_path)
print(f"Saved → {out_path}")